In [1]:
import requests
from bs4 import BeautifulSoup as soup
import pandas as pd

In [11]:
def get_spread_page():
    url = "https://rotogrinders.com/lineups/nfl"
    html = requests.get(url).text
    page = soup(html)
    return page
page = get_spread_page()

team_list = page.find_all("span", {"class": "shrt"})
teams = [i.text for i in team_list]
odds_list = page.find_all("a", {"href": "/nfl/odds"})
odds = [i.text for i in odds_list]
teams.sort()
print(teams)

['ARI', 'ATL', 'BAL', 'BUF', 'CAR', 'CHI', 'CIN', 'CLE', 'DAL', 'DEN', 'DET', 'GBP', 'HOU', 'IND', 'JAC', 'KCC', 'LAC', 'LAR', 'LVR', 'MIA', 'MIN', 'NEP', 'NOS', 'NYG', 'NYJ', 'PHI', 'PIT', 'SEA', 'SFO', 'TBB', 'TEN', 'WAS']


In [22]:
replace_dict = {"SFO":"SF", "NOS":"NO", "GBP":"GB", "KCC": "KC", "NOS": "NO", "SFO":"SF"}
clean_teams = [replace_dict.get(item,item) for item in teams]
print(clean_teams)

['ARI', 'ATL', 'BAL', 'BUF', 'CAR', 'CHI', 'CIN', 'CLE', 'DAL', 'DEN', 'DET', 'GB', 'HOU', 'IND', 'JAC', 'KC', 'LAC', 'LAR', 'LVR', 'MIA', 'MIN', 'NEP', 'NO', 'NYG', 'NYJ', 'PHI', 'PIT', 'SEA', 'SF', 'TBB', 'TEN', 'WAS']


In [23]:
df_odds = dict(zip(clean_teams, odds))
print(df_odds)

{'ARI': '19.50', 'ATL': '26.50', 'BAL': '17.25', 'BUF': '24.25', 'CAR': '30.50', 'CHI': '23.50', 'CIN': '14.75', 'CLE': '32.25', 'DAL': '23.75', 'DEN': '27.75', 'DET': '26.25', 'GB': '25.25', 'HOU': '20.00', 'IND': '22.50', 'JAC': '25.25', 'KC': '19.25', 'LAC': '19.00', 'LAR': '22.00', 'LVR': '24.50', 'MIA': '23.00', 'MIN': '24.75', 'NEP': '29.25', 'NO': '25.00', 'NYG': '27.00', 'NYJ': '19.25', 'PHI': '25.75', 'PIT': '22.50', 'SEA': '21.50', 'SF': '28.00', 'TBB': '21.50', 'TEN': '24.00', 'WAS': '27.50'}


In [19]:
df_dk = pd.read_csv("data/nfl/DKSalaries20211003.csv")
dk_teams = df_dk.TeamAbbrev.unique()
dk_teams.sort()
dk_teams

array(['ARI', 'ATL', 'BAL', 'BUF', 'CAR', 'CHI', 'CLE', 'DAL', 'DEN',
       'DET', 'GB', 'HOU', 'IND', 'KC', 'LAR', 'MIA', 'MIN', 'NO', 'NYG',
       'NYJ', 'PHI', 'PIT', 'SEA', 'SF', 'TEN', 'WAS'], dtype=object)

In [6]:
df_dk.columns

Index(['Position', 'Name + ID', 'Name', 'ID', 'Roster Position', 'Salary',
       'Game Info', 'TeamAbbrev', 'AvgPointsPerGame'],
      dtype='object')

In [40]:
df_dk["expts"] = df_dk["TeamAbbrev"].map(df_odds)
df_dk.expts = df_dk.expts.astype('float')
pts_bins = [1.1, 1.05, 1.0, .95,.9]
df_dk["rkpts"] = pd.cut(df_dk.expts, 5, labels=pts_bins)
df_dk.rkpts = df_dk.rkpts.astype('float')
df_dk["new_expts"] = df_dk.AvgPointsPerGame * df_dk.rkpts
df_dk.head()

,Position,Name + ID,Name,ID,Roster Position,Salary,Game Info,TeamAbbrev,AvgPointsPerGame,expts,rkpts,new_expts
0,RB,Derrick Henry (19394143),Derrick Henry,19394143,RB/FLEX,8800,TEN@NYJ 10/03/2021 01:00PM ET,TEN,27.93,24.00,1.0,27.930
1,RB,Christian McCaffrey (19394145),Christian McCaffrey,19394145,RB/FLEX,8700,CAR@DAL 10/03/2021 01:00PM ET,CAR,19.47,30.50,0.9,17.523
2,RB,Alvin Kamara (19394147),Alvin Kamara,19394147,RB/FLEX,8400,NYG@NO 10/03/2021 01:00PM ET,NO,15.30,25.00,1.0,15.300
3,QB,Patrick Mahomes (19394053),Patrick Mahomes,19394053,QB,8100,KC@PHI 10/03/2021 01:00PM ET,KC,29.73,19.25,1.1,32.703
4,TE,Travis Kelce (19395157),Travis Kelce,19395157,TE/FLEX,8100,KC@PHI 10/03/2021 01:00PM ET,KC,24.30,19.25,1.1,26.730


In [44]:
df_dk.AvgPointsPerGame = df_dk.new_expts
df_dk = df_dk.drop(['expts', 'rkpts', 'new_expts'], axis=1)
df_dk

,Position,Name + ID,Name,ID,Roster Position,Salary,Game Info,TeamAbbrev,AvgPointsPerGame
0,RB,Derrick Henry (19394143),Derrick Henry,19394143,RB/FLEX,8800,TEN@NYJ 10/03/2021 01:00PM ET,TEN,27.930
1,RB,Christian McCaffrey (19394145),Christian McCaffrey,19394145,RB/FLEX,8700,CAR@DAL 10/03/2021 01:00PM ET,CAR,17.523
2,RB,Alvin Kamara (19394147),Alvin Kamara,19394147,RB/FLEX,8400,NYG@NO 10/03/2021 01:00PM ET,NO,15.300
3,QB,Patrick Mahomes (19394053),Patrick Mahomes,19394053,QB,8100,KC@PHI 10/03/2021 01:00PM ET,KC,32.703
4,TE,Travis Kelce (19395157),Travis Kelce,19395157,TE/FLEX,8100,KC@PHI 10/03/2021 01:00PM ET,KC,26.730
...,...,...,...,...,...,...,...,...,...
800,DST,Falcons (19395542),Falcons,19395542,DST,2300,WAS@ATL 10/03/2021 01:00PM ET,ATL,1.900
801,DST,Lions (19395543),Lions,19395543,DST,2200,DET@CHI 10/03/2021 01:00PM ET,DET,2.330
802,DST,Eagles (19395544),Eagles,19395544,DST,2100,KC@PHI 10/03/2021 01:00PM ET,PHI,6.330
803,DST,Texans (19395546),Texans,19395546,DST,2000,HOU@BUF 10/03/2021 01:00PM ET,HOU,5.500
